# By_County Data Processing Notebook

This notebook extracts and processes county-level demographic and socioeconomic data from the American Community Survey (ACS) to support an analysis comparing population characteristics with a Medicaid fraud dataset. Because the fraud dataset spans only three years (2021–2023), ACS 5-Year Estimates are used for each corresponding year. The ACS 5-Year product provides the most reliable statistics for small geographic areas—such as counties—due to its larger sample size and lower variance.

The county level is used as the primary geographic unit for this analysis because it provides a broad yet meaningful regional overview and aligns with the structure of the fraud dataset. A separate ZIP-code-level extraction will also be produced for more granular follow-up work.

## Key Variables Extracted

### **Demographic Indicators**
- TotalPopulation  
- Race and ethnicity percentages:  
  - Pct_White  
  - Pct_Black  
  - Pct_Asian  
  - Pct_Hispanic  

### **Economic Indicators**
- MedianIncome  
- LaborForce  
- Unemployed  
- UnemploymentRate  

### **Poverty Measures**
- PovertyTotal  
- PovertyUniverse  
- PovertyRate  

### **Educational Attainment**
- HighSchoolOrHigher  
- BachelorsOrHigher

In [1]:
import sys
import requests
import pandas as pd
import numpy as np

First, I went to the https://api.census.gov/data/key_signup.html to get a free API Key for the US Census Bureau. I have also share API Key below to help with the configuration.

In [2]:
#Configuration

API_KEY = "6755c585a386003e31f7c5540c276cd03714bb6d"
YEAR = "2021"
STATE_FIPS = None   # None = all states

if not API_KEY or API_KEY.strip() == "":
    sys.exit("ERROR: Missing Census API Key.")

## ACS Education Variable Summary

The following variables are drawn from **ACS Table B15003: Educational Attainment for the Population 25 Years and Over**, representing counts of individuals aged 25+ with specific education levels:

- **B15003_001** — Total population age 25+  

**High school or equivalent:**
- B15003_017 — High school graduate (regular diploma)  
- B15003_018 — GED or alternative credential  
- B15003_019 — Some college, less than 1 year  
- B15003_020 — Some college, 1 or more years, no degree  
- B15003_021 — Associate’s degree  

**Bachelor’s degree or higher:**
- B15003_022 — Bachelor’s degree  
- B15003_023 — Master’s degree  
- B15003_024 — Professional school degree  
- B15003_025 — Doctorate degree  

These variables are used to compute two derived indicators:
- **HighSchoolOrHigher:** sum of B15003_017–B15003_025  
- **BachelorsOrHigher:** sum of B15003_022–B15003_025  

In [3]:
# VARIABLES TO DOWNLOAD

VARIABLES = {
    # Population
    "TotalPopulation": "B01003_001E",
    
    # Income
    "MedianIncome": "B19013_001E",
    
    # Poverty (total in poverty / total population)
    "PovertyTotal": "B17001_002E",
    "PovertyUniverse": "B17001_001E",

    # Race
    "White": "B02001_002E",
    "Black": "B02001_003E",
    "Asian": "B02001_005E",
    "Hispanic": "B03003_003E",

    # Education (Age 25 and above)
    # High school or higher = sum of B15003_017E to B15003_025E
    # Bachelor’s or higher = sum of B15003_022E to B15003_025E
    "Edu_HS_17": "B15003_017E",
    "Edu_HS_18": "B15003_018E",
    "Edu_HS_19": "B15003_019E",
    "Edu_HS_20": "B15003_020E",
    "Edu_HS_21": "B15003_021E",
    "Edu_BA_22": "B15003_022E",
    "Edu_BA_23": "B15003_023E",
    "Edu_BA_24": "B15003_024E",
    "Edu_BA_25": "B15003_025E",

    # Unemployment
    "LaborForce": "B23025_003E",
    "Unemployed": "B23025_005E"
}

---

## API Configuration and Data Extraction

The notebook retrieves ACS5 data for each year (2021, 2022, and 2023) using a Census API key obtained from the Census Bureau’s API portal. For each year:

- A request is constructed to retrieve population, income, poverty, race, education, and labor-force indicators.
- The response is converted into a pandas DataFrame.
- Variable names are replaced with descriptive labels to improve readability.
- Derived metrics—such as poverty rate, unemployment rate, and educational attainment—are calculated.
- Data quality checks ensure no rows contain missing, empty, or invalid values.

In [4]:
# Build GET parameter string
get_vars = "NAME," + ",".join(VARIABLES.values())

# API REQUEST
base_url = f"https://api.census.gov/data/{YEAR}/acs/acs5"

params = {
    "get": get_vars,
    "for": "county:*",
    "key": API_KEY
}

if STATE_FIPS:
    params["in"] = f"state:{STATE_FIPS}"

print("Requesting Census API...")
resp = requests.get(base_url, params=params)

resp.raise_for_status()

try:
    data = resp.json()
except:
    sys.exit("ERROR: Could not decode JSON response from Census API.")

Requesting Census API...


In [5]:
# CONVERT TO DATAFRAME

df = pd.DataFrame(data[1:], columns=data[0])

# Convert numeric columns
for col in VARIABLES.values():
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Split NAME into County and State
df[["CountyName", "StateName"]] = df["NAME"].str.split(", ", n=1, expand=True)

df.head()

,NAME,B01003_001E,B19013_001E,B17001_002E,B17001_001E,B02001_002E,B02001_003E,B02001_005E,B03003_003E,B15003_017E,...,B15003_022E,B15003_023E,B15003_024E,B15003_025E,B23025_003E,B23025_005E,state,county,CountyName,StateName
0,"Autauga County, Alabama",58239,62660,7847,57790,43755,11470,647,1775,10458,...,6507,3649,524,464,26623,752,01,001,Autauga County,Alabama
1,"Baldwin County, Alabama",227131,64346,20598,223772,192034,19895,2175,10634,36186,...,33379,13884,3059,2240,108361,3994,01,003,Baldwin County,Alabama
2,"Barbour County, Alabama",25259,36422,5890,22250,11495,11985,106,1176,5204,...,1212,520,164,111,9369,808,01,005,Barbour County,Alabama
3,"Bibb County, Alabama",22412,54277,3558,21000,17020,5003,46,634,5556,...,1276,476,93,68,9107,884,01,007,Bibb County,Alabama
4,"Blount County, Alabama",58884,52830,7720,58323,54439,760,216,5612,11019,...,3783,1812,286,180,25798,1554,01,009,Blount County,Alabama


In [6]:
df["B17001_002E"].head()

0     7847
1    20598
2     5890
3     3558
4     7720
Name: B17001_002E, dtype: int64

Rename the variables to an understandable name

In [7]:
df = df.rename(columns={
    "B01003_001E": "TotalPopulation",
    "B19013_001E": "MedianIncome",
    "B17001_002E": "PovertyTotal",
    "B17001_001E": "PovertyUniverse",
    "B02001_002E": "White",
    "B02001_003E": "Black",
    "B02001_005E": "Asian",
    "B03003_003E": "Hispanic",
    "B23025_003E": "LaborForce",
    "B23025_005E": "Unemployed",
    "state": "StateFIPS",
    "county": "CountyFIPS",

    # ---- Education variables (B15003 table) ----
    "B15003_017E": "Edu_HS_17",
    "B15003_018E": "Edu_HS_18",
    "B15003_019E": "Edu_HS_19",
    "B15003_020E": "Edu_HS_20",
    "B15003_021E": "Edu_HS_21",
    "B15003_022E": "Edu_BA_22",
    "B15003_023E": "Edu_BA_23",
    "B15003_024E": "Edu_BA_24",
    "B15003_025E": "Edu_BA_25"
})

In [8]:
# Ensure FIPS codes are zero-padded
df["StateFIPS"] = df["StateFIPS"].astype(str).str.zfill(2)
df["CountyFIPS"] = df["CountyFIPS"].astype(str).str.zfill(3)

# Full 5-digit County FIPS
df["CountyFIPS_5"] = df["StateFIPS"] + df["CountyFIPS"]

In [9]:
df.head()

,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,Edu_HS_17,...,Edu_BA_23,Edu_BA_24,Edu_BA_25,LaborForce,Unemployed,StateFIPS,CountyFIPS,CountyName,StateName,CountyFIPS_5
0,"Autauga County, Alabama",58239,62660,7847,57790,43755,11470,647,1775,10458,...,3649,524,464,26623,752,01,001,Autauga County,Alabama,01001
1,"Baldwin County, Alabama",227131,64346,20598,223772,192034,19895,2175,10634,36186,...,13884,3059,2240,108361,3994,01,003,Baldwin County,Alabama,01003
2,"Barbour County, Alabama",25259,36422,5890,22250,11495,11985,106,1176,5204,...,520,164,111,9369,808,01,005,Barbour County,Alabama,01005
3,"Bibb County, Alabama",22412,54277,3558,21000,17020,5003,46,634,5556,...,476,93,68,9107,884,01,007,Bibb County,Alabama,01007
4,"Blount County, Alabama",58884,52830,7720,58323,54439,760,216,5612,11019,...,1812,286,180,25798,1554,01,009,Blount County,Alabama,01009


In [10]:
df["PovertyTotal"]

0        7847
1       20598
2        5890
3        3558
4        7720
        ...  
3216    23520
3217     4425
3218     9881
3219    16335
3220    16048
Name: PovertyTotal, Length: 3221, dtype: int64

Calculating the derived variables

In [11]:
# CREATE DERIVED VARIABLES

# Poverty Rate
df["PovertyRate"] = df["PovertyTotal"] / df["PovertyUniverse"]

# Race Percentages
df["Pct_White"] = df["White"] / df["TotalPopulation"]
df["Pct_Black"] = df["Black"] / df["TotalPopulation"]
df["Pct_Asian"] = df["Asian"] / df["TotalPopulation"]
df["Pct_Hispanic"] = df["Hispanic"] / df["TotalPopulation"]

# Education Levels
df["HighSchoolOrHigher"] = (
    df["Edu_HS_17"] + df["Edu_HS_18"] + df["Edu_HS_19"] +
    df["Edu_HS_20"] + df["Edu_HS_21"] +
    df["Edu_BA_22"] + df["Edu_BA_23"] + df["Edu_BA_24"] + df["Edu_BA_25"]
)

df["BachelorsOrHigher"] = (
    df["Edu_BA_22"] + df["Edu_BA_23"] +
    df["Edu_BA_24"] + df["Edu_BA_25"]
)

# Unemployment Rate
df["UnemploymentRate"] = df["Unemployed"] / df["LaborForce"]

In [12]:
df.head()

,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,Edu_HS_17,...,StateName,CountyFIPS_5,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate
0,"Autauga County, Alabama",58239,62660,7847,57790,43755,11470,647,1775,10458,...,Alabama,01001,0.135785,0.751301,0.196947,0.011109,0.030478,35488,11144,0.028246
1,"Baldwin County, Alabama",227131,64346,20598,223772,192034,19895,2175,10634,36186,...,Alabama,01003,0.092049,0.845477,0.087593,0.009576,0.046819,147422,52562,0.036858
2,"Barbour County, Alabama",25259,36422,5890,22250,11495,11985,106,1176,5204,...,Alabama,01005,0.264719,0.455085,0.474484,0.004197,0.046558,13617,2007,0.086242
3,"Bibb County, Alabama",22412,54277,3558,21000,17020,5003,46,634,5556,...,Alabama,01007,0.169429,0.759415,0.223229,0.002052,0.028288,12932,1913,0.097068
4,"Blount County, Alabama",58884,52830,7720,58323,54439,760,216,5612,11019,...,Alabama,01009,0.132366,0.924513,0.012907,0.003668,0.095306,34018,6061,0.060237


In [13]:
# Condition 1: has NaN anywhere
cond_nan = df.isna().any(axis=1)

# Condition 2: has empty string anywhere
cond_empty = df.apply(lambda row: row.astype(str).str.strip().eq('')).any(axis=1)

# Condition 3: has '-' anywhere
cond_dash = df.apply(lambda row: row.astype(str).str.strip().eq('-')).any(axis=1)

print(df[cond_nan | cond_empty | cond_dash])

Empty DataFrame
Columns: [NAME, TotalPopulation, MedianIncome, PovertyTotal, PovertyUniverse, White, Black, Asian, Hispanic, Edu_HS_17, Edu_HS_18, Edu_HS_19, Edu_HS_20, Edu_HS_21, Edu_BA_22, Edu_BA_23, Edu_BA_24, Edu_BA_25, LaborForce, Unemployed, StateFIPS, CountyFIPS, CountyName, StateName, CountyFIPS_5, PovertyRate, Pct_White, Pct_Black, Pct_Asian, Pct_Hispanic, HighSchoolOrHigher, BachelorsOrHigher, UnemploymentRate]
Index: []

[0 rows x 33 columns]


In [14]:
df2021 = df

All the data are clean and without zero, empty or negative numbers.

### Processing Subsequent Years (2022 and 2023)

After completing the full extraction, cleaning, and computation steps for the 2021 ACS dataset, the same procedure is repeated for the next two years—2022 and 2023. This ensures that each year is processed using an identical workflow, allowing for consistent comparisons across time.

For both 2022 and 2023, the notebook:

- Sends a new API request for the corresponding year.
- Converts the returned JSON response into a pandas DataFrame.
- Applies the same variable renaming and derived metric calculations used for 2021.
- Performs the same data quality checks to ensure completeness and validity.

By replicating the exact steps for each year, the final datasets remain structurally aligned, making them suitable for year-to-year comparison and later merging into a combined panel dataset.

In [15]:
#Configuration

API_KEY = "6755c585a386003e31f7c5540c276cd03714bb6d"
YEAR = "2022"
STATE_FIPS = None   # None = all states

if not API_KEY or API_KEY.strip() == "":
    sys.exit("ERROR: Missing Census API Key.")


# VARIABLES TO DOWNLOAD
VARIABLES = {
    # Population
    "TotalPopulation": "B01003_001E",
    
    # Income
    "MedianIncome": "B19013_001E",
    
    # Poverty (total in poverty / total population)
    "PovertyTotal": "B17001_002E",
    "PovertyUniverse": "B17001_001E",

    # Race
    "White": "B02001_002E",
    "Black": "B02001_003E",
    "Asian": "B02001_005E",
    "Hispanic": "B03003_003E",

    # Education (Age 25 and above)
    # High school or higher = sum of B15003_017E to B15003_025E
    # Bachelor’s or higher = sum of B15003_022E to B15003_025E
    "Edu_HS_17": "B15003_017E",
    "Edu_HS_18": "B15003_018E",
    "Edu_HS_19": "B15003_019E",
    "Edu_HS_20": "B15003_020E",
    "Edu_HS_21": "B15003_021E",
    "Edu_BA_22": "B15003_022E",
    "Edu_BA_23": "B15003_023E",
    "Edu_BA_24": "B15003_024E",
    "Edu_BA_25": "B15003_025E",

    # Unemployment
    "LaborForce": "B23025_003E",
    "Unemployed": "B23025_005E"
}

# Build GET parameter string
get_vars = "NAME," + ",".join(VARIABLES.values())

# API REQUEST
base_url = f"https://api.census.gov/data/{YEAR}/acs/acs5"

params = {
    "get": get_vars,
    "for": "county:*",
    "key": API_KEY
}

if STATE_FIPS:
    params["in"] = f"state:{STATE_FIPS}"

print("Requesting Census API...")
resp = requests.get(base_url, params=params)

resp.raise_for_status()

try:
    data = resp.json()
except:
    sys.exit("ERROR: Could not decode JSON response from Census API.")
    
# CONVERT TO DATAFRAME

df = pd.DataFrame(data[1:], columns=data[0])

# Convert numeric columns
for col in VARIABLES.values():
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Split NAME into County and State
df[["CountyName", "StateName"]] = df["NAME"].str.split(", ", n=1, expand=True)

df = df.rename(columns={
    "B01003_001E": "TotalPopulation",
    "B19013_001E": "MedianIncome",
    "B17001_002E": "PovertyTotal",
    "B17001_001E": "PovertyUniverse",
    "B02001_002E": "White",
    "B02001_003E": "Black",
    "B02001_005E": "Asian",
    "B03003_003E": "Hispanic",
    "B23025_003E": "LaborForce",
    "B23025_005E": "Unemployed",
    "state": "StateFIPS",
    "county": "CountyFIPS",

    # ---- Education variables (B15003 table) ----
    "B15003_017E": "Edu_HS_17",
    "B15003_018E": "Edu_HS_18",
    "B15003_019E": "Edu_HS_19",
    "B15003_020E": "Edu_HS_20",
    "B15003_021E": "Edu_HS_21",
    "B15003_022E": "Edu_BA_22",
    "B15003_023E": "Edu_BA_23",
    "B15003_024E": "Edu_BA_24",
    "B15003_025E": "Edu_BA_25"
})

# Ensure FIPS codes are zero-padded
df["StateFIPS"] = df["StateFIPS"].astype(str).str.zfill(2)
df["CountyFIPS"] = df["CountyFIPS"].astype(str).str.zfill(3)

# Full 5-digit County FIPS
df["CountyFIPS_5"] = df["StateFIPS"] + df["CountyFIPS"]

# CREATE DERIVED VARIABLES

# Poverty Rate
df["PovertyRate"] = df["PovertyTotal"] / df["PovertyUniverse"]

# Race Percentages
df["Pct_White"] = df["White"] / df["TotalPopulation"]
df["Pct_Black"] = df["Black"] / df["TotalPopulation"]
df["Pct_Asian"] = df["Asian"] / df["TotalPopulation"]
df["Pct_Hispanic"] = df["Hispanic"] / df["TotalPopulation"]

# Education Levels
df["HighSchoolOrHigher"] = (
    df["Edu_HS_17"] + df["Edu_HS_18"] + df["Edu_HS_19"] +
    df["Edu_HS_20"] + df["Edu_HS_21"] +
    df["Edu_BA_22"] + df["Edu_BA_23"] + df["Edu_BA_24"] + df["Edu_BA_25"]
)

df["BachelorsOrHigher"] = (
    df["Edu_BA_22"] + df["Edu_BA_23"] +
    df["Edu_BA_24"] + df["Edu_BA_25"]
)

# Unemployment Rate
df["UnemploymentRate"] = df["Unemployed"] / df["LaborForce"]

df

Requesting Census API...


,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,Edu_HS_17,...,StateName,CountyFIPS_5,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate
0,"Autauga County, Alabama",58761,68315,6630,58291,43747,11496,658,1864,10001,...,Alabama,01001,0.113740,0.744490,0.195640,0.011198,0.031722,36331,11879,0.027685
1,"Baldwin County, Alabama",233420,71039,23445,229539,195998,19445,2046,11210,38059,...,Alabama,01003,0.102140,0.839680,0.083305,0.008765,0.048025,152991,54385,0.034435
2,"Barbour County, Alabama",24877,39712,5280,21851,11309,11668,126,1202,5136,...,Alabama,01005,0.241637,0.454597,0.469028,0.005065,0.048318,13520,2100,0.057538
3,"Bibb County, Alabama",22251,50669,4297,20836,16872,4603,69,650,5261,...,Alabama,01007,0.206230,0.758258,0.206867,0.003101,0.029212,12559,1739,0.087062
4,"Blount County, Alabama",59077,57440,8277,58399,53941,729,100,5721,11248,...,Alabama,01009,0.141732,0.913063,0.012340,0.001693,0.096840,33370,6017,0.059696
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3217,"Vega Baja Municipio, Puerto Rico",54182,23701,23265,53847,21954,2444,9,53023,10997,...,Puerto Rico,72145,0.432057,0.405190,0.045107,0.000166,0.978609,30586,10001,0.163425
3218,"Vieques Municipio, Puerto Rico",8199,17062,4433,8199,1813,579,16,7744,2556,...,Puerto Rico,72147,0.540676,0.221125,0.070618,0.001951,0.944505,4349,875,0.131436
3219,"Villalba Municipio, Puerto Rico",21984,22461,9342,21858,8732,1889,0,21905,4882,...,Puerto Rico,72149,0.427395,0.397198,0.085926,0.000000,0.996406,12257,3358,0.139260
3220,"Yabucoa Municipio, Puerto Rico",30313,19972,15070,30293,2393,11349,10,30252,5536,...,Puerto Rico,72151,0.497475,0.078943,0.374394,0.000330,0.997988,16823,4147,0.111134


## Comparison Across Years

After each yearly dataset is processed, the notebook performs cross-year validation to identify meaningful differences. This includes:

- Aligning counties by name, including handling duplicate county names where necessary.
- Identifying variables that differ across years.
- Generating a consolidated comparison table summarizing all detected changes.

This ensures that differences across years reflect real changes in ACS estimates, rather than formatting inconsistencies or data extraction errors.


In [16]:
# --- prepare copies so we don't modify originals
a = df.copy()
b = df2021.copy()

# --- detect duplicate NAMEs and disambiguate if necessary
if a['NAME'].duplicated().any() or b['NAME'].duplicated().any():
    # append an occurrence count to duplicate names to make them unique indices
    a['__dup_idx'] = a.groupby('NAME').cumcount().astype(str).radd('_').replace({'_0': ''})
    b['__dup_idx'] = b.groupby('NAME').cumcount().astype(str).radd('_').replace({'_0': ''})
    a['__NAME_unique'] = a['NAME'] + a['__dup_idx']
    b['__NAME_unique'] = b['NAME'] + b['__dup_idx']
    idx_col = '__NAME_unique'
else:
    idx_col = 'NAME'

# --- set index by NAME (or disambiguated name)
a_idx = a.set_index(idx_col)
b_idx = b.set_index(idx_col)

# --- find names present in both datasets
common_names = a_idx.index.intersection(b_idx.index)

# --- restrict to the common subset
a_common = a_idx.loc[common_names].sort_index()
b_common = b_idx.loc[common_names].sort_index()

# --- mask of rows where any column differs
row_diff_mask = (a_common != b_common).any(axis=1)

# --- basic counts
n_total_common = len(common_names)
n_value_diffs = row_diff_mask.sum()

print(f"Names present in both: {n_total_common}")
print(f"Names with value differences: {n_value_diffs}")

# --- side-by-side combined dataframe (values from df and df0)
left = a_common[row_diff_mask].add_suffix('_df')
right = b_common[row_diff_mask].add_suffix('_df0')
combined = pd.concat([left, right], axis=1)

# --- numeric columns difference (df - df0) for numeric fields only
numeric_cols = a_common.select_dtypes(include=[np.number]).columns.intersection(
               b_common.select_dtypes(include=[np.number]).columns)
if len(numeric_cols) > 0:
    diff_numeric = (a_common[numeric_cols] - b_common[numeric_cols]).loc[row_diff_mask]
    diff_numeric = diff_numeric.add_suffix('_diff')
    report = pd.concat([combined, diff_numeric], axis=1)
else:
    report = combined

# --- optional: list which non-equal columns changed per row
cols_changed = (a_common != b_common).loc[row_diff_mask].apply(lambda row: list(row[row].index), axis=1)
report['columns_changed'] = cols_changed

# --- if we used a unique helper column, restore original NAME column for readability
if idx_col != 'NAME':
    # extract original name before any "_index" suffix
    report.insert(0, 'NAME_original', report.index.to_series().str.split('_').str[0])

# --- preview and save option
print("\nPreview of differences (first 10 rows):")
display(report.head(10))   # in Jupyter this will show a nice table

Names present in both: 3213
Names with value differences: 3213

Preview of differences (first 10 rows):


,TotalPopulation_df,MedianIncome_df,PovertyTotal_df,PovertyUniverse_df,White_df,Black_df,Asian_df,Hispanic_df,Edu_HS_17_df,Edu_HS_18_df,...,Unemployed_diff,PovertyRate_diff,Pct_White_diff,Pct_Black_diff,Pct_Asian_diff,Pct_Hispanic_diff,HighSchoolOrHigher_diff,BachelorsOrHigher_diff,UnemploymentRate_diff,columns_changed
NAME,,,,,,,,,,,,,,,,,,,,,
"Abbeville County, South Carolina",24368,49759,3752,23589,16915,6372,58,441,5620,1060,...,-134,-0.012917,-0.002742,-0.007238,-0.000205,0.001071,263,148,-0.012676,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
"Acadia Parish, Louisiana",57674,44977,13459,56735,45230,9511,95,1780,13809,2765,...,-306,0.006448,-0.002912,-0.000932,-0.000226,0.001997,-272,1,-0.010293,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
"Accomack County, Virginia",33367,52694,5240,32960,20860,9484,268,3084,6992,1070,...,-19,-0.010105,-0.019884,-0.001109,0.000394,0.001316,-206,75,-0.001451,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
"Ada County, Idaho",497494,83881,42460,484201,429014,6260,12867,44317,53631,13835,...,-485,-0.003979,-0.014045,-0.000406,0.000172,0.002108,9839,10694,-0.002773,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
"Adair County, Iowa",7479,63172,736,7304,7192,48,9,196,1999,175,...,-31,-0.014253,-0.009069,0.000638,-0.007803,0.001338,58,-22,-0.007960,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
"Adair County, Kentucky",18887,49690,3490,17640,17444,307,73,468,3975,1089,...,6,0.015511,-0.007689,-0.006738,0.000254,0.001627,218,-18,-0.000817,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
"Adair County, Missouri",25299,51020,5438,23055,22803,1043,472,702,3933,533,...,26,0.006674,-0.005799,0.002038,-0.005811,0.000951,175,240,0.002992,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
"Adair County, Oklahoma",19726,44955,4339,19439,7756,66,268,1504,4874,1084,...,63,-0.022113,-0.003103,-0.000838,0.002749,0.004467,30,25,0.007813,"[TotalPopulation, MedianIncome, PovertyTotal, ..."
"Adams County, Colorado",520149,86297,49625,515939,342214,18215,20574,215119,77514,19887,...,454,0.000279,-0.052263,0.000997,-0.000314,0.004093,3616,3633,0.000512,"[TotalPopulation, MedianIncome, PovertyTotal, ..."


In [17]:
df2021[df2021['NAME'] == "Abbeville County, South Carolina"]

,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,Edu_HS_17,...,StateName,CountyFIPS_5,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate
2317,"Abbeville County, South Carolina",24374,45710,4072,23678,16986,6550,63,415,5350,...,South Carolina,45001,0.171974,0.69689,0.268729,0.002585,0.017026,14334,3121,0.041017


In [18]:
df[df['NAME'] == "Abbeville County, South Carolina"]

,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,Edu_HS_17,...,StateName,CountyFIPS_5,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate
2318,"Abbeville County, South Carolina",24368,49759,3752,23589,16915,6372,58,441,5620,...,South Carolina,45001,0.159057,0.694148,0.26149,0.00238,0.018098,14597,3269,0.028341


In [19]:
df2022 = df

In [20]:
#Configuration

API_KEY = "6755c585a386003e31f7c5540c276cd03714bb6d"
YEAR = "2023"
STATE_FIPS = None   # None = all states

if not API_KEY or API_KEY.strip() == "":
    sys.exit("ERROR: Missing Census API Key.")


# VARIABLES TO DOWNLOAD
VARIABLES = {
    # Population
    "TotalPopulation": "B01003_001E",
    
    # Income
    "MedianIncome": "B19013_001E",
    
    # Poverty (total in poverty / total population)
    "PovertyTotal": "B17001_002E",
    "PovertyUniverse": "B17001_001E",

    # Race
    "White": "B02001_002E",
    "Black": "B02001_003E",
    "Asian": "B02001_005E",
    "Hispanic": "B03003_003E",

    # Education (Age 25 and above)
    # High school or higher = sum of B15003_017E to B15003_025E
    # Bachelor’s or higher = sum of B15003_022E to B15003_025E
    "Edu_HS_17": "B15003_017E",
    "Edu_HS_18": "B15003_018E",
    "Edu_HS_19": "B15003_019E",
    "Edu_HS_20": "B15003_020E",
    "Edu_HS_21": "B15003_021E",
    "Edu_BA_22": "B15003_022E",
    "Edu_BA_23": "B15003_023E",
    "Edu_BA_24": "B15003_024E",
    "Edu_BA_25": "B15003_025E",


    # Unemployment
    "LaborForce": "B23025_003E",
    "Unemployed": "B23025_005E"
}

# Build GET parameter string
get_vars = "NAME," + ",".join(VARIABLES.values())

# API REQUEST
base_url = f"https://api.census.gov/data/{YEAR}/acs/acs5"

params = {
    "get": get_vars,
    "for": "county:*",
    "key": API_KEY
}

if STATE_FIPS:
    params["in"] = f"state:{STATE_FIPS}"

print("Requesting Census API...")
resp = requests.get(base_url, params=params)

resp.raise_for_status()

try:
    data = resp.json()
except:
    sys.exit("ERROR: Could not decode JSON response from Census API.")
    
# CONVERT TO DATAFRAME

df = pd.DataFrame(data[1:], columns=data[0])

# Convert numeric columns
for col in VARIABLES.values():
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Split NAME into County and State
df[["CountyName", "StateName"]] = df["NAME"].str.split(", ", n=1, expand=True)

df = df.rename(columns={
    "B01003_001E": "TotalPopulation",
    "B19013_001E": "MedianIncome",
    "B17001_002E": "PovertyTotal",
    "B17001_001E": "PovertyUniverse",
    "B02001_002E": "White",
    "B02001_003E": "Black",
    "B02001_005E": "Asian",
    "B03003_003E": "Hispanic",
    "B23025_003E": "LaborForce",
    "B23025_005E": "Unemployed",
    "state": "StateFIPS",
    "county": "CountyFIPS",

    # ---- Education variables (B15003 table) ----
    "B15003_017E": "Edu_HS_17",
    "B15003_018E": "Edu_HS_18",
    "B15003_019E": "Edu_HS_19",
    "B15003_020E": "Edu_HS_20",
    "B15003_021E": "Edu_HS_21",
    "B15003_022E": "Edu_BA_22",
    "B15003_023E": "Edu_BA_23",
    "B15003_024E": "Edu_BA_24",
    "B15003_025E": "Edu_BA_25"
})

# Ensure FIPS codes are zero-padded
df["StateFIPS"] = df["StateFIPS"].astype(str).str.zfill(2)
df["CountyFIPS"] = df["CountyFIPS"].astype(str).str.zfill(3)

# Full 5-digit County FIPS
df["CountyFIPS_5"] = df["StateFIPS"] + df["CountyFIPS"]

# CREATE DERIVED VARIABLES

# Poverty Rate
df["PovertyRate"] = df["PovertyTotal"] / df["PovertyUniverse"]

# Race Percentages
df["Pct_White"] = df["White"] / df["TotalPopulation"]
df["Pct_Black"] = df["Black"] / df["TotalPopulation"]
df["Pct_Asian"] = df["Asian"] / df["TotalPopulation"]
df["Pct_Hispanic"] = df["Hispanic"] / df["TotalPopulation"]

# Education Levels
df["HighSchoolOrHigher"] = (
    df["Edu_HS_17"] + df["Edu_HS_18"] + df["Edu_HS_19"] +
    df["Edu_HS_20"] + df["Edu_HS_21"] +
    df["Edu_BA_22"] + df["Edu_BA_23"] + df["Edu_BA_24"] + df["Edu_BA_25"]
)

df["BachelorsOrHigher"] = (
    df["Edu_BA_22"] + df["Edu_BA_23"] +
    df["Edu_BA_24"] + df["Edu_BA_25"]
)

# Unemployment Rate
df["UnemploymentRate"] = df["Unemployed"] / df["LaborForce"]

df

Requesting Census API...


,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,Edu_HS_17,...,StateName,CountyFIPS_5,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate
0,"Autauga County, Alabama",59285,69841,6275,58731,43616,11829,616,2188,11084,...,Alabama,01001,0.106843,0.735700,0.199528,0.010390,0.036906,36804,11530,0.025416
1,"Baldwin County, Alabama",239945,75019,24819,236041,198721,19144,2272,13393,38757,...,Alabama,01003,0.105147,0.828194,0.079785,0.009469,0.055817,157767,56408,0.031943
2,"Barbour County, Alabama",24757,44290,4746,21650,10891,11616,136,1490,5317,...,Alabama,01005,0.219215,0.439916,0.469201,0.005493,0.060185,13717,2021,0.057086
3,"Bibb County, Alabama",22152,51215,4258,20759,16634,4587,51,744,5449,...,Alabama,01007,0.205116,0.750903,0.207069,0.002302,0.033586,12799,1827,0.099840
4,"Blount County, Alabama",59292,61096,8269,58665,53062,747,51,5962,11610,...,Alabama,01009,0.140953,0.894927,0.012599,0.000860,0.100553,33898,6386,0.058352
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3217,"Vega Baja Municipio, Puerto Rico",54058,23877,22724,53745,13681,2249,10,53036,11002,...,Puerto Rico,72145,0.422811,0.253080,0.041603,0.000185,0.981094,31145,10025,0.134242
3218,"Vieques Municipio, Puerto Rico",8147,17531,4850,8147,1028,222,17,7803,2656,...,Puerto Rico,72147,0.595311,0.126181,0.027249,0.002087,0.957776,4207,745,0.085572
3219,"Villalba Municipio, Puerto Rico",21778,24882,9065,21670,7552,2219,0,21700,4509,...,Puerto Rico,72149,0.418320,0.346772,0.101892,0.000000,0.996418,12287,3503,0.135114
3220,"Yabucoa Municipio, Puerto Rico",29868,21279,14826,29821,2001,5900,11,29732,5489,...,Puerto Rico,72151,0.497166,0.066995,0.197536,0.000368,0.995447,16712,4119,0.096726


In [21]:
df2023 = df


## Merging and Exporting the Final Dataset

Finally:

- The datasets for 2021, 2022, and 2023 are each assigned a `Year` column.
- All yearly datasets are stacked vertically into a single panel dataset.
- The merged dataset is sorted by county name.
- The final combined dataset is exported as **merged_census_2021_2023.csv**.

This comprehensive dataset is used to link ACS demographic and socioeconomic indicators with the Missouri Medicaid fraud dataset, enabling deeper statistical and machine-learning analysis.

In [22]:
# Add a 'Year' column to each dataframe
df2021['Year'] = 2021
df2022['Year'] = 2022
df2023['Year'] = 2023

# Concatenate them vertically
merged_df = pd.concat([df2021, df2022, df2023], ignore_index=True)

# Sort by NAME
merged_df = merged_df.sort_values(by='NAME').reset_index(drop=True)

# Optional: preview
merged_df.head(10)


,NAME,TotalPopulation,MedianIncome,PovertyTotal,PovertyUniverse,White,Black,Asian,Hispanic,Edu_HS_17,...,CountyFIPS_5,PovertyRate,Pct_White,Pct_Black,Pct_Asian,Pct_Hispanic,HighSchoolOrHigher,BachelorsOrHigher,UnemploymentRate,Year
0,"Abbeville County, South Carolina",24352,52112,3318,23529,16882,6287,51,475,5485,...,45001,0.141017,0.693249,0.258172,0.002094,0.019506,14860,3332,0.033900,2023
1,"Abbeville County, South Carolina",24374,45710,4072,23678,16986,6550,63,415,5350,...,45001,0.171974,0.696890,0.268729,0.002585,0.017026,14334,3121,0.041017,2021
2,"Abbeville County, South Carolina",24368,49759,3752,23589,16915,6372,58,441,5620,...,45001,0.159057,0.694148,0.261490,0.002380,0.018098,14597,3269,0.028341,2022
3,"Acadia Parish, Louisiana",57218,45266,13750,56402,44296,8983,111,1802,13744,...,22001,0.243786,0.774162,0.156996,0.001940,0.031494,30473,5071,0.078069,2023
4,"Acadia Parish, Louisiana",57674,44977,13459,56735,45230,9511,95,1780,13809,...,22001,0.237226,0.784236,0.164910,0.001647,0.030863,30496,5217,0.087750,2022
5,"Acadia Parish, Louisiana",58200,42368,13209,57237,45812,9652,109,1680,13475,...,22001,0.230777,0.787148,0.165842,0.001873,0.028866,30768,5216,0.098043,2021
6,"Accomack County, Virginia",33388,50601,5593,33078,21537,9527,255,3042,7443,...,51001,0.169085,0.645052,0.285342,0.007637,0.091111,20105,5133,0.038800,2021
7,"Accomack County, Virginia",33326,57500,4883,32848,20240,9077,285,3528,7577,...,51001,0.148654,0.607334,0.272370,0.008552,0.105863,20181,5314,0.038151,2023
8,"Accomack County, Virginia",33367,52694,5240,32960,20860,9484,268,3084,6992,...,51001,0.158981,0.625169,0.284233,0.008032,0.092427,19899,5208,0.037349,2022
9,"Ada County, Idaho",497494,83881,42460,484201,429014,6260,12867,44317,53631,...,16001,0.087691,0.862350,0.012583,0.025864,0.089080,324035,145388,0.032220,2022


In [23]:
merged_df.to_csv('merged_census_2021_2023.csv', index=False)